## 1. Portada del proyecto
Presentación del objetivo y alcance del análisis para Liverpool Padres Junio 2026.


In [211]:
# Pendiente por implementar en la siguiente fase
pass


## 2. Importación de librerías
Espacio para declarar las librerías permitidas del proyecto.


In [212]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import matplotlib.pyplot as plt


## 3. Definición de rutas
Definición de rutas relativas con pathlib para garantizar portabilidad.


In [213]:
BASE_DIR = Path.cwd()
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
LOGOS_DIR = BASE_DIR / "logotipos"
OUTPUT_FILE = DATA_PROCESSED / "Liverpool_VN_Aniv_Oct_26_consolidado_2.0_limpio.csv"

# Crear carpeta de salida si no existe
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)


## 4. Carga de los 8 CSV
Bloque para leer los 8 archivos CSV de entrada con pandas.read_csv().
Incluye Liverpool_MTY VALLE_Padres_Jun_26

In [214]:

# Diccionario con codigo de tienda, archivo y sucursal esperada
archivos_tiendas = {
    'AZP': {'archivo': 'Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv', 'sucursal': 'ATIZAPAN'},
    'COA': {'archivo': 'Liverpool_COAPA_VN_Aniv_Oct_26.csv', 'sucursal': 'COAPA'},
    'ECA': {'archivo': 'Liverpool_Ecatepec_VN_Aniv_Oct_26.csv', 'sucursal': 'ECATEPEC'},
    'TOR': {'archivo': 'Liverpool_Toreo_VN_Aniv_Oct_26.csv', 'sucursal': 'TOREO'},
    'CDJ': {'archivo': 'Liverpool_CDJardin_VN_Aniv_Oct_26.csv', 'sucursal': 'CD JARDIN'},
    'LIN': {'archivo': 'Liverpool_Lindavista_VN_Aniv_Oct_26.csv', 'sucursal': 'LINDAVISTA'},
    'INT': {'archivo': 'Liverpool_INTERLOMAS_VN_Aniv_Oct_26.csv', 'sucursal': 'INTERLOMAS'},
    'PDT': {'archivo': 'Liverpool_Parque_Delta_VN_Aniv_Oct_26.csv', 'sucursal': 'DELTA'}    
}

# Carga de cada CSV en un diccionario de DataFrames con fallback de encoding
dataframes = {}
for codigo, info in archivos_tiendas.items():
    ruta_csv = DATA_RAW / info['archivo']
    try:
        df_tmp = pd.read_csv(ruta_csv, encoding='utf-8')
    except UnicodeDecodeError:
        df_tmp = pd.read_csv(ruta_csv, encoding='latin-1')
    dataframes[codigo] = df_tmp

print(f'Total de DataFrames cargados: {len(dataframes)}')
list(dataframes.keys())


Total de DataFrames cargados: 8


['AZP', 'COA', 'ECA', 'TOR', 'CDJ', 'LIN', 'INT', 'PDT']

## 5. Revisión rápida de estructura
Validación inicial de columnas, tipos y forma de cada DataFrame.


In [215]:
# Revisión rápida de estructura por tienda
for codigo, df_tmp in dataframes.items():
    print('=' * 80)
    print(f'Tienda: {codigo} | Sucursal esperada: {archivos_tiendas[codigo]["sucursal"]}')
    print(f'Shape: {df_tmp.shape}')
    print('Columns:')
    print(df_tmp.columns.tolist())
    print('Dtypes:')
    print(df_tmp.dtypes)
    print('Head():')
    print(df_tmp.head())
    print('isna().sum():')
    print(df_tmp.isna().sum())
    print()


Tienda: AZP | Sucursal esperada: ATIZAPAN
Shape: (36, 8)
Columns:
['ID', 'Marca', 'Producto', 'Frase', 'Genero', 'Fecha', 'Sucursal', 'Horario']
Dtypes:
ID           int64
Marca       object
Producto    object
Frase       object
Genero      object
Fecha       object
Sucursal    object
Horario     object
dtype: object
Head():
   ID             Marca  Producto              Frase Genero       Fecha  \
0   1          GIVENCHY  Perfume              RAFAEL      H  02/10/2026   
1   2  CAROLINA HERRERA  Perfume   FELIZ ANIVERSARIO      H  02/10/2026   
2   3      RALPH LAUREN  Perfume              RAFAEL      H  02/10/2026   
3   4          GIVENCHY  Perfume                 SOL      M  02/10/2026   
4   5              DIOR  Perfume             GEDIDIA      M  02/10/2026   

   Sucursal      Horario  
0  ATIZAPAN  01:00 p. m.  
1  ATIZAPAN          NaN  
2  ATIZAPAN          NaN  
3  ATIZAPAN  06:00 p. m.  
4  ATIZAPAN          NaN  
isna().sum():
ID          0
Marca       0
Producto    0
Fras

## 6. Corrección de nombres de columnas
Corrección de Surcursal a Sucursal y estandarización a snake_case.


In [216]:
import re

def normalizar_columnas(df):
    # 1) Quitar espacios en nombres de columnas
    cols = [str(col).strip() for col in df.columns]

    # 2) Cambiar Surcursal a Sucursal si existe
    cols = ['Sucursal' if col == 'Surcursal' else col for col in cols]

    # 3) Convertir nombres de columnas a min?sculas
    cols = [col.lower() for col in cols]

    # 4) Reemplazar espacios por guiones bajos
    cols = [col.replace(' ', '_') for col in cols]

    # 5) Dejar nombres en snake_case
    cols = [re.sub(r'[^a-z0-9_]', '_', col) for col in cols]
    cols = [re.sub(r'_+', '_', col).strip('_') for col in cols]

    df.columns = cols
    return df

# Aplicar normalización a cada DataFrame cargado
for codigo, df_tmp in dataframes.items():
    dataframes[codigo] = normalizar_columnas(df_tmp)

# Vista rápida de columnas normalizadas
for codigo, df_tmp in dataframes.items():
    print(codigo, df_tmp.columns.tolist())


AZP ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
COA ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
ECA ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
TOR ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
CDJ ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
LIN ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
INT ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']
PDT ['id', 'marca', 'producto', 'frase', 'genero', 'fecha', 'sucursal', 'horario']


## 7. Agregar trazabilidad por tienda
Creación de id_global, codigo_tienda, archivo_origen e id_trazabilidad.


In [217]:
# Agregar trazabilidad por tienda en cada DataFrame
for codigo, df_tmp in dataframes.items():
    archivo = archivos_tiendas[codigo]['archivo']

    # id_local consecutivo por tienda
    df_tmp['id_local'] = range(1, len(df_tmp) + 1)

    # Metadatos de trazabilidad
    df_tmp['archivo_origen'] = archivo
    df_tmp['codigo_tienda'] = codigo

    # Formato solicitado: ATZ-00001
    df_tmp['id_trazabilidad'] = df_tmp['codigo_tienda'] + '-' + df_tmp['id_local'].astype(str).str.zfill(5)

    dataframes[codigo] = df_tmp

# Revisi?n r?pida de trazabilidad
for codigo, df_tmp in dataframes.items():
    print('=' * 80)
    print(f'Tienda: {codigo}')
    print(df_tmp[['codigo_tienda', 'id_local', 'id_trazabilidad', 'archivo_origen']].head())


Tienda: AZP
  codigo_tienda  id_local id_trazabilidad  \
0           AZP         1       AZP-00001   
1           AZP         2       AZP-00002   
2           AZP         3       AZP-00003   
3           AZP         4       AZP-00004   
4           AZP         5       AZP-00005   

                          archivo_origen  
0  Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv  
1  Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv  
2  Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv  
3  Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv  
4  Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv  
Tienda: COA
  codigo_tienda  id_local id_trazabilidad                      archivo_origen
0           COA         1       COA-00001  Liverpool_COAPA_VN_Aniv_Oct_26.csv
1           COA         2       COA-00002  Liverpool_COAPA_VN_Aniv_Oct_26.csv
2           COA         3       COA-00003  Liverpool_COAPA_VN_Aniv_Oct_26.csv
3           COA         4       COA-00004  Liverpool_COAPA_VN_Aniv_Oct_26.csv
4           COA         5       COA-00005  Liverpool_COAP

## 9. Unión de los 8 DataFrames
Consolidación de los datos en un único DataFrame.


In [218]:
# Unión de los 8 DataFrames
lista_df = [dataframes[codigo] for codigo in archivos_tiendas.keys()]
df_liverpool = pd.concat(lista_df, ignore_index=True)

# Trazabilidad global
df_liverpool['id_global'] = np.arange(1, len(df_liverpool) + 1)
df_liverpool['grabado'] = 1

print('Shape consolidado:', df_liverpool.shape)
df_liverpool.head(20)


Shape consolidado: (391, 14)


,id,marca,producto,frase,genero,fecha,sucursal,horario,id_local,archivo_origen,codigo_tienda,id_trazabilidad,id_global,grabado
0,1,GIVENCHY,Perfume,RAFAEL,H,02/10/2026,ATIZAPAN,01:00 p. m.,1,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00001,1,1
1,2,CAROLINA HERRERA,Perfume,FELIZ ANIVERSARIO,H,02/10/2026,ATIZAPAN,NaN,2,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00002,2,1
2,3,RALPH LAUREN,Perfume,RAFAEL,H,02/10/2026,ATIZAPAN,NaN,3,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00003,3,1
3,4,GIVENCHY,Perfume,SOL,M,02/10/2026,ATIZAPAN,06:00 p. m.,4,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00004,4,1
4,5,DIOR,Perfume,GEDIDIA,M,02/10/2026,ATIZAPAN,NaN,5,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00005,5,1
5,6,CAROLINA HERRERA,Perfume,FELIZ ANIVERSARIO,M,02/10/2026,ATIZAPAN,07:00 p. m.,6,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00006,6,1
6,7,VALENTINO,Perfume,TE AMO,M,02/10/2026,ATIZAPAN,NaN,7,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00007,7,1
7,8,CARLO CORINTO,Perfume,MARY,M,02/10/2026,ATIZAPAN,08:00 p. m.,8,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00008,8,1
8,9,MARC JACOBS,Perfume,MELISSA,M,03/10/2026,ATIZAPAN,02:00 p. m.,9,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00009,9,1
9,10,YSL,Perfume,IKER CARBAJAL,H,03/10/2026,ATIZAPAN,04:00 p. m.,10,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00010,10,1


## 9. Limpieza básica permitida
Aplicación exclusiva de reglas permitidas sin modificar la columna frase.


In [219]:
# Limpieza b?sica permitida
# 1) marca
df_liverpool['marca'] = df_liverpool['marca'].astype('string').str.strip()
df_liverpool['marca'] = df_liverpool['marca'].replace('', pd.NA).fillna('SIN MARCA').str.upper()
# Homologaci?n espec?fica de marcas
mapa_marcas = {
    'BENETHON': 'BENETTON','MOSSIMO': 'MOSCHINO', 'ADOLFO DOMÍNGUEZ': 'ADOLFO DOMINGUEZ', 'CLARANCE' : 'CLARINS',
    'CALVIN KLEIN': 'CALVIN KLEIN',
    'CALVIN  KLEIN': 'CALVIN KLEIN',
    'DOLCE & GABANA': 'DOLCE&GABBANA',
    'DOLCHE & GABANNA': 'DOLCE&GABBANA',
    'DOLCHE & GABBANA': 'DOLCE&GABBANA',
    'DOLCE & GABANNA': 'DOLCE&GABBANA',
    'DOLCE & GABBANA' : 'DOLCE&GABBANA',
    'PERRI ELIS': 'PERRY ELLIS',
    'PERRY ELIS': 'PERRY ELLIS',
    'PERRY ELLIES': 'PERRY ELLIS',
    'PERRY LLIS': 'PERRY ELLIS',
    'GIORGIO ARMANI': 'ARMANI',
    'YVES SAINT LAURENT': 'YSL',
    'MOSSCHINO': 'MOSCHINO',
    'PACO RABBANE': 'PACO RABANNE',
    'MONT BLACK': 'MONTBLANC',
    'MONT BLANC': 'MONTBLANC',
    'JADORE': 'DIOR',
    "J'ADORE": 'DIOR',
    'J ADORE': 'DIOR',
    'JADORE DIOR': 'DIOR',
    "J'ADORE DIOR": 'DIOR',
    'J?ADORE': 'DIOR',
    'J?ADORE DIOR': 'DIOR','TOMMY' : 'TOMMY HILFIGER', 'GUERLINE' : 'GUERLAIN', 'GUERLIAN' : 'GUERLAIN'
}
df_liverpool['marca'] = df_liverpool['marca'].replace(mapa_marcas)
# 2) producto
df_liverpool['producto'] = df_liverpool['producto'].astype('string').str.strip()
df_liverpool['producto'] = df_liverpool['producto'].replace('', pd.NA).fillna('SIN PRODUCTO')
# 3) genero
def homologar_genero(valor):
    if pd.isna(valor):
        return 'SIN DATO'
    v = str(valor).strip().upper()
    if v == '':
        return 'SIN DATO'
    if v in {'H', 'HOMBRE', 'MASCULINO'}:
        return 'HOMBRE'
    if v in {'M', 'MUJER', 'F', 'FEMENINO'}:
        return 'MUJER'
    if v in {'SIN DATO', 'NO ESPECIFICADO', 'NO APLICA', 'N/A', 'NA'}:
        return 'SIN DATO'
    return 'OTRO'
df_liverpool['genero'] = df_liverpool['genero'].apply(homologar_genero)
# 4) fecha
fecha_texto = df_liverpool['fecha'].astype('string').str.strip()
fecha_1 = pd.to_datetime(fecha_texto, format='%d-%m-%Y', errors='coerce')
fecha_2 = pd.to_datetime(fecha_texto, format='%d-%m-%y', errors='coerce')
fecha_3 = pd.to_datetime(fecha_texto, format='%d/%m/%Y', errors='coerce')
fecha_4 = pd.to_datetime(fecha_texto, format='%d/%m/%y', errors='coerce')
df_liverpool['fecha'] = fecha_1.fillna(fecha_2).fillna(fecha_3).fillna(fecha_4)
# Correcci?n espec?fica solicitada: 2025-04-24 -> 2026-04-24
df_liverpool.loc[df_liverpool['fecha'] == pd.Timestamp('2025-04-24'), 'fecha'] = pd.Timestamp('2026-04-24')
# 5) sucursal
df_liverpool['sucursal'] = df_liverpool['sucursal'].astype('string').str.strip()
df_liverpool['sucursal'] = df_liverpool['sucursal'].replace('', pd.NA).fillna('SIN SUCURSAL').str.upper()
# Nota: La columna frase se conserva tal como viene (sin modificar).
df_liverpool.head()


,id,marca,producto,frase,genero,fecha,sucursal,horario,id_local,archivo_origen,codigo_tienda,id_trazabilidad,id_global,grabado
0,1,GIVENCHY,Perfume,RAFAEL,HOMBRE,2026-10-02,ATIZAPAN,01:00 p. m.,1,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00001,1,1
1,2,CAROLINA HERRERA,Perfume,FELIZ ANIVERSARIO,HOMBRE,2026-10-02,ATIZAPAN,NaN,2,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00002,2,1
2,3,RALPH LAUREN,Perfume,RAFAEL,HOMBRE,2026-10-02,ATIZAPAN,NaN,3,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00003,3,1
3,4,GIVENCHY,Perfume,SOL,MUJER,2026-10-02,ATIZAPAN,06:00 p. m.,4,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00004,4,1
4,5,DIOR,Perfume,GEDIDIA,MUJER,2026-10-02,ATIZAPAN,NaN,5,Liverpool_ATIZAPAN_VN_Aniv_Oct_26.csv,AZP,AZP-00005,5,1


## 10. Revisión de nulos
Identificación de nulos sin eliminar registros.


In [220]:
# Crear formato datetime para la columna fecha. NOTA: Mejora
df_liverpool['fecha'] = pd.to_datetime(
    df_liverpool['fecha'],
    errors='coerce',
    dayfirst=True
)

# Revisi?n de nulos
nulos_por_columna = df_liverpool.isna().sum()
print('Nulos por columna:')
print(nulos_por_columna)

porcentaje_nulos = (df_liverpool.isna().mean() * 100).round(2)
print('\nPorcentaje de nulos por columna (%):')
print(porcentaje_nulos)

resumen_nulos = pd.DataFrame({
    'columna': nulos_por_columna.index,
    'nulos': nulos_por_columna.values,
    'porcentaje_nulos': porcentaje_nulos.values
}).sort_values('porcentaje_nulos', ascending=False)

resumen_nulos


Nulos por columna:
id                   0
marca                0
producto             0
frase                0
genero               0
fecha                0
sucursal             0
horario            242
id_local             0
archivo_origen       0
codigo_tienda        0
id_trazabilidad      0
id_global            0
grabado              0
dtype: int64

Porcentaje de nulos por columna (%):
id                  0.00
marca               0.00
producto            0.00
frase               0.00
genero              0.00
fecha               0.00
sucursal            0.00
horario            61.89
id_local            0.00
archivo_origen      0.00
codigo_tienda       0.00
id_trazabilidad     0.00
id_global           0.00
grabado             0.00
dtype: float64


,columna,nulos,porcentaje_nulos
7,horario,242,61.89
0,id,0,0.00
2,producto,0,0.00
3,frase,0,0.00
4,genero,0,0.00
1,marca,0,0.00
5,fecha,0,0.00
6,sucursal,0,0.00
8,id_local,0,0.00
9,archivo_origen,0,0.00


## 11. Revisión de duplicados sin eliminarlos
Detección y reporte de duplicados sin borrarlos.


In [221]:
# Revisi?n de duplicados sin eliminarlos
total_registros = len(df_liverpool)
total_duplicados = int(df_liverpool.duplicated().sum())
porcentaje_duplicados = round((total_duplicados / total_registros) * 100, 2) if total_registros > 0 else 0.0

print(f'Total de registros: {total_registros}')
print(f'Total de duplicados: {total_duplicados}')
print(f'Porcentaje de duplicados: {porcentaje_duplicados}%')

muestra_duplicados = df_liverpool[df_liverpool.duplicated(keep=False)].head(10)
print('\nMuestra de posibles duplicados (hasta 10 filas):')
muestra_duplicados


Total de registros: 391
Total de duplicados: 0
Porcentaje de duplicados: 0.0%

Muestra de posibles duplicados (hasta 10 filas):


,id,marca,producto,frase,genero,fecha,sucursal,horario,id_local,archivo_origen,codigo_tienda,id_trazabilidad,id_global,grabado


## 12. Guardado del CSV limpio
Guardado del consolidado limpio en data/processed cuando se autorice.


In [222]:
# Guardado del CSV limpio
df_liverpool.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Archivo guardado en: {OUTPUT_FILE}')


Archivo guardado en: c:\Users\moise\OneDrive\DATA SCIENCE - TRIPLE TEN\LIVERPOOL\Dashboard_Liverpool_VN_Aniv_Oct_26\notebooks\data\processed\Liverpool_VN_Aniv_Oct_26_consolidado_2.0_limpio.csv


## 13. Agrupaciones principales
C?lculo de agrupaciones por tienda, marca, fecha y g?nero.


In [223]:
# Agrupaciones principales

# 1) Grabados por tienda (todas las tiendas)
df_grabados_por_tienda = (
    df_liverpool.groupby('sucursal', dropna=False)['grabado']
    .sum()
    .reset_index()
)
df_grabados_por_tienda.columns = ['sucursal', 'total_grabados']
df_grabados_por_tienda = df_grabados_por_tienda.sort_values('total_grabados', ascending=False)

# 2) Grabados por marca (todas las marcas)
df_grabados_por_marca = (
    df_liverpool.groupby('marca', dropna=False)['grabado']
    .sum()
    .reset_index()
)
df_grabados_por_marca.columns = ['marca', 'total_grabados']
df_grabados_por_marca = df_grabados_por_marca.sort_values('total_grabados', ascending=False)

# 3) Grabados por fecha (solo rango de 3 días: 19-21 junio 2026)
rango_inicio = pd.Timestamp('2026-06-19')
rango_fin = pd.Timestamp('2026-06-21')

df_rango_3dias = df_liverpool[(df_liverpool['fecha'] >= rango_inicio) & (df_liverpool['fecha'] <= rango_fin)].copy()

df_grabados_por_fecha = (
    df_rango_3dias.assign(fecha=df_rango_3dias['fecha'].dt.normalize())
    .groupby('fecha', dropna=False)['grabado']
    .sum()
    .reset_index()
)
df_grabados_por_fecha.columns = ['fecha', 'total_grabados']
df_grabados_por_fecha = df_grabados_por_fecha.sort_values('fecha', ascending=True)

mapa_dias = {
    pd.Timestamp('2026-06-19'): 'Viernes',
    pd.Timestamp('2026-06-20'): 'Sábado',
    pd.Timestamp('2026-06-21'): 'Domingo'
}
df_grabados_por_fecha['dia_nombre'] = df_grabados_por_fecha['fecha'].map(mapa_dias)

# 3.1) Tabla: grabados por tienda por día, NOTA: INICIO NUEVO FRAGMENTO CODIGO
df_grabados_por_tienda_dia = (
    df_rango_3dias.assign(fecha=df_rango_3dias['fecha'].dt.normalize())
    .groupby(['fecha', 'sucursal'], dropna=False)['grabado']
    .sum()
    .reset_index()
)

df_grabados_por_tienda_dia.columns = ['fecha', 'sucursal', 'total_grabados']

df_grabados_por_tienda_dia['dia_nombre'] = df_grabados_por_tienda_dia['fecha'].map(mapa_dias)

df_grabados_por_tienda_dia = df_grabados_por_tienda_dia.sort_values(
    'total_grabados',
    ascending=False
)

print(df_grabados_por_tienda_dia.head(10))

# 3.1) Tabla: grabados por tienda por día, NOTA: FIN NUEVO FRAGMENTO CODIGO

# Total del rango de 3 días (sumando las 9 tiendas)
total_grabados_3dias = int(df_grabados_por_fecha['total_grabados'].sum())

# 4) Género total
df_genero_total = (
    df_liverpool.groupby('genero', dropna=False)['grabado']
    .sum()
    .reset_index()
)
df_genero_total.columns = ['genero', 'total_grabados']

# 5) G?nero por tienda
df_genero_por_tienda = (
    df_liverpool.groupby(['sucursal', 'genero'], dropna=False)['grabado']
    .sum()
    .reset_index()
)
df_genero_por_tienda.columns = ['sucursal', 'genero', 'total_grabados']

# 6) G?nero por marca
df_genero_por_marca = (
    df_liverpool.groupby(['marca', 'genero'], dropna=False)['grabado']
    .sum()
    .reset_index()
)
df_genero_por_marca.columns = ['marca', 'genero', 'total_grabados']

print('Agrupaciones creadas correctamente.')
print(f'Total grabados en rango 19-21 junio 2026: {total_grabados_3dias}')
df_grabados_por_fecha



Empty DataFrame
Columns: [fecha, sucursal, total_grabados, dia_nombre]
Index: []
Agrupaciones creadas correctamente.
Total grabados en rango 19-21 junio 2026: 0


,fecha,total_grabados,dia_nombre


## 14. Gráficos de barras
Visualizaciones de conteos con gráficos de barras.


In [224]:
# Gráficos de barras con Plotly

fig_tienda = px.bar(
    df_grabados_por_tienda,
    x='sucursal',
    y='total_grabados',
    text='total_grabados',
    title='Grabados por tienda (todas las tiendas)',
    labels={'sucursal': 'Sucursal', 'total_grabados': 'Total de grabados'}
)
fig_tienda.update_traces(textposition='outside')
fig_tienda.show()

fig_marca = px.bar(
    df_grabados_por_marca,
    x='marca',
    y='total_grabados',
    text='total_grabados',
    title='Grabados por marca (todas las marcas)',
    labels={'marca': 'Marca', 'total_grabados': 'Total de grabados'}
)
fig_marca.update_traces(textposition='outside')
fig_marca.show()

# Histograma por fecha para el rango de 3 dáas (19 al 21 junio 2026)
# Renombrando fechas a nombres de día: Viernes, Sábado y Domingo
df_rango_3dias_plot = df_rango_3dias.copy()
mapa_dias_plot = {
    pd.Timestamp('2026-06-19'): 'Viernes',
    pd.Timestamp('2026-06-20'): 'Sábado',
    pd.Timestamp('2026-06-21'): 'Domingo'
}
df_rango_3dias_plot['dia_nombre'] = df_rango_3dias_plot['fecha'].dt.normalize().map(mapa_dias_plot)

fig_fecha_hist = px.histogram(
    df_rango_3dias_plot,
    x='dia_nombre',
    y='grabado',
    histfunc='sum',
    category_orders={'dia_nombre': ['Viernes', 'Sábado', 'Domingo']},
    title='Histograma de grabados por día (19 al 21 junio 2026)',
    labels={'dia_nombre': 'Día', 'grabado': 'Total de grabados'}
)
fig_fecha_hist.update_traces(texttemplate='%{y}', textposition='outside')
fig_fecha_hist.show()


## 15. Gráficas de pastel
Visualizaciones de proporciones de género con gráficas de pastel.


In [225]:
# Gráficas de pastel para género con Plotly

fig_genero_total = px.pie(
    df_genero_total,
    names='genero',
    values='total_grabados',
    title='Distribución de género total'
)
fig_genero_total.show()

tienda_ejemplo = df_grabados_por_tienda['sucursal'].iloc[0]
df_genero_tienda_ejemplo = df_genero_por_tienda[df_genero_por_tienda['sucursal'] == tienda_ejemplo]
fig_genero_tienda = px.pie(
    df_genero_tienda_ejemplo,
    names='genero',
    values='total_grabados',
    title=f'Distribución de género en tienda: {tienda_ejemplo}'
)
fig_genero_tienda.show()

marca_ejemplo = df_grabados_por_marca['marca'].iloc[0]
df_genero_marca_ejemplo = df_genero_por_marca[df_genero_por_marca['marca'] == marca_ejemplo]
fig_genero_marca = px.pie(
    df_genero_marca_ejemplo,
    names='genero',
    values='total_grabados',
    title=f'Distribución de género en marca: {marca_ejemplo}'
)
fig_genero_marca.show()


## 16. Dashboard ejecutivo en notebook
Resumen ejecutivo con indicadores y visualizaciones clave.


In [226]:
# Dashboard ejecutivo en notebook (Top 5 permitido)

top5_tiendas = df_grabados_por_tienda.head(5)
top5_marcas = df_grabados_por_marca.head(5)

fig_top5_tiendas = px.bar(
    top5_tiendas,
    x='sucursal',
    y='total_grabados',
    title='Top 5 tiendas por grabados',
    labels={'sucursal': 'Sucursal', 'total_grabados': 'Total de grabados'}
)
fig_top5_tiendas.show()

fig_top5_marcas = px.bar(
    top5_marcas,
    x='marca',
    y='total_grabados',
    title='Top 5 marcas por grabados',
    labels={'marca': 'Marca', 'total_grabados': 'Total de grabados'}
)
fig_top5_marcas.show()

resumen_ejecutivo = {
    'total_grabados': int(df_liverpool['grabado'].sum()),
    'total_tiendas': int(df_grabados_por_tienda['sucursal'].nunique()),
    'total_marcas': int(df_grabados_por_marca['marca'].nunique()),
    'tienda_lider': df_grabados_por_tienda.iloc[0]['sucursal'],
    'marca_lider': df_grabados_por_marca.iloc[0]['marca'],
    }

resumen_ejecutivo


{'total_grabados': 391,
 'total_tiendas': 8,
 'total_marcas': 51,
 'tienda_lider': 'TOREO',
 'marca_lider': 'LANCOME'}

## 17. Conclusiones
Síntesis final basada en resultados reales del análisis.


In [227]:
# Conclusiones basadas en el DataFrame consolidado
tienda_lider = df_grabados_por_tienda.iloc[0]
marca_lider = df_grabados_por_marca.iloc[0]
fecha_lider = df_grabados_por_fecha.sort_values('total_grabados', ascending=False).iloc[0]
genero_lider = df_genero_total.sort_values('total_grabados', ascending=False).iloc[0]

print('Conclusiones:')
print(f'- Tienda líder: {tienda_lider["sucursal"]} con {int(tienda_lider["total_grabados"])} grabados.')
print(f'- Marca líder: {marca_lider["marca"]} con {int(marca_lider["total_grabados"])} grabados.')
print(f'- Fecha con más grabados: {fecha_lider["fecha"]} con {int(fecha_lider["total_grabados"])} grabados.')
print(f'- Género predominante: {genero_lider["genero"]} con {int(genero_lider["total_grabados"])} grabados.')


IndexError: single positional indexer is out-of-bounds

## 18. Análisis emocional de frases
Se crean columnas derivadas para analizar patrones emocionales sin modificar la columna original `frase`.


In [228]:
import re
import unicodedata

# Diccionarios editables de patrones emocionales
PATRONES = {
    'PAPA': [
        'papa', 'papá', 'papi', 'papito', 'padre', 'pa', 'abuelo', 'abuelito',
        'mi papa', 'para papa', 'te amo papa', 'feliz dia papa','feliz día papa',
        'dia de los padres', 'día de los padres','el mejor papa del mundo'
    ],
    'AMOR': [
        'amor', 'te amo', 'te quiero', 'mi amor',
        'corazon', 'cariño', 'mi vida', 'para ti', 'siempre juntos', 'familia'
    ],
    'PASION': [
        'pasion', 'deseo', 'fuego', 'intenso', 'intensa', 'besos',
        'seduccion', 'inolvidable', 'atrevete'
    ],
    'FORMALIDAD': [
        'felicidades', 'gracias', 'con aprecio', 'con respeto', 'estimada',
        'estimado', 'sra', 'sr', 'lic', 'dra', 'dr', 'de parte de'
    ],
    'LIBERTAD': [
        'libre', 'libertad', 'vuela', 'sueña', 'sueños', 'aventura', 'viaja',
        'sin límites', 'se tu', 'vive', 'volar', 'alas'
    ]
}

def normalizar_frase(texto):
    """Normaliza texto para búsqueda de patrones sin tocar frase original."""
    if pd.isna(texto):
        return ''
    s = str(texto).lower()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def score_categoria(frase_norm, patrones):
    """Cuenta coincidencias simples por patrón en una frase normalizada."""
    score = 0
    for patron in patrones:
        patron_norm = normalizar_frase(patron)
        if patron_norm and patron_norm in frase_norm:
            score += 1
    return score

def parece_personal_regalo(frase_norm):
    """Regla simple para personalizaciones, nombres o detalles individuales."""
    if not frase_norm:
        return False

    palabras = frase_norm.split()

    if 1 <= len(palabras) <= 3:
        return True

    if frase_norm.startswith('para '):
        return True

    if re.fullmatch(r'[a-z](\s+[a-z]){0,3}', frase_norm):
        return True

    return False

def clasificar_emocion(row):
    # Prioridad solicitada: PAPA antes de AMOR
    if row['score_papa'] > 0:
        return 'PAPÁ'

    scores = {
        'AMOR': row['score_amor'],
        'PASION': row['score_pasion'],
        'FORMALIDAD': row['score_formalidad'],
        'LIBERTAD': row['score_libertad']
    }

    max_score = max(scores.values())

    if max_score == 0:
        if row['score_personal_regalo'] > 0:
            return 'PERSONAL / REGALO'
        return 'SIN CLASIFICAR'

    categorias_max = [k for k, v in scores.items() if v == max_score and v > 0]

    if len(categorias_max) > 1:
        return 'MIXTO'

    return categorias_max[0]

# Crear columnas derivadas sin modificar frase original
df_liverpool['frase_normalizada'] = df_liverpool['frase'].apply(normalizar_frase)
df_liverpool['score_papa'] = df_liverpool['frase_normalizada'].apply(lambda x: score_categoria(x, PATRONES['PAPA']))
df_liverpool['score_amor'] = df_liverpool['frase_normalizada'].apply(lambda x: score_categoria(x, PATRONES['AMOR']))
df_liverpool['score_pasion'] = df_liverpool['frase_normalizada'].apply(lambda x: score_categoria(x, PATRONES['PASION']))
df_liverpool['score_formalidad'] = df_liverpool['frase_normalizada'].apply(lambda x: score_categoria(x, PATRONES['FORMALIDAD']))
df_liverpool['score_libertad'] = df_liverpool['frase_normalizada'].apply(lambda x: score_categoria(x, PATRONES['LIBERTAD']))
df_liverpool['score_personal_regalo'] = df_liverpool['frase_normalizada'].apply(lambda x: int(parece_personal_regalo(x)))
df_liverpool['categoria_emocional'] = df_liverpool.apply(clasificar_emocion, axis=1)

df_liverpool[[
    'frase', 'frase_normalizada', 'score_papa', 'score_amor', 'score_pasion',
    'score_formalidad', 'score_libertad', 'score_personal_regalo', 'categoria_emocional'
]].head(10)


,frase,frase_normalizada,score_papa,score_amor,score_pasion,score_formalidad,score_libertad,score_personal_regalo,categoria_emocional
0,RAFAEL,rafael,0,0,0,0,0,1,PERSONAL / REGALO
1,FELIZ ANIVERSARIO,feliz aniversario,0,0,0,0,0,1,PERSONAL / REGALO
2,RAFAEL,rafael,0,0,0,0,0,1,PERSONAL / REGALO
3,SOL,sol,0,0,0,0,0,1,PERSONAL / REGALO
4,GEDIDIA,gedidia,0,0,0,0,0,1,PERSONAL / REGALO
5,FELIZ ANIVERSARIO,feliz aniversario,0,0,0,0,0,1,PERSONAL / REGALO
6,TE AMO,te amo,0,1,0,0,0,1,AMOR
7,MARY,mary,0,0,0,0,0,1,PERSONAL / REGALO
8,MELISSA,melissa,0,0,0,0,0,1,PERSONAL / REGALO
9,IKER CARBAJAL,iker carbajal,0,0,0,0,0,1,PERSONAL / REGALO


## 19. Validación de clasificación emocional
Se generan tablas de control para revisar distribución y ejemplos por categoría emocional.


In [229]:
# 1) Conteo por categoria_emocional
conteo_categoria_emocional = df_liverpool['categoria_emocional'].value_counts(dropna=False).rename_axis('categoria_emocional').reset_index()
conteo_categoria_emocional.columns = ['categoria_emocional', 'conteo']
conteo_categoria_emocional

# 2) Porcentaje por categoria_emocional
porcentaje_categoria_emocional = (
    df_liverpool['categoria_emocional'].value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
    .rename_axis('categoria_emocional')
    .reset_index()
)
porcentaje_categoria_emocional.columns = ['categoria_emocional', 'porcentaje']
porcentaje_categoria_emocional

# 3) Muestra de frases por cada categoria_emocional
muestra_frases_categoria = (
    df_liverpool[['categoria_emocional', 'frase']]
    .dropna(subset=['frase'])
    .groupby('categoria_emocional', group_keys=False)
    .head(5)
    .reset_index(drop=True)
)
muestra_frases_categoria

# 4) Conteo de SIN CLASIFICAR
conteo_sin_clasificar = int((df_liverpool['categoria_emocional'] == 'SIN CLASIFICAR').sum())
print('Conteo SIN CLASIFICAR:', conteo_sin_clasificar)

# 5) Conteo de MIXTO
conteo_mixto = int((df_liverpool['categoria_emocional'] == 'MIXTO').sum())
print('Conteo MIXTO:', conteo_mixto)

# 6) Conteo de MAM?
conteo_mama = int((df_liverpool['categoria_emocional'] == 'PAPA').sum())
print('Conteo PAPA:', conteo_mama)


Conteo SIN CLASIFICAR: 5
Conteo MIXTO: 0
Conteo PAPA: 0


## 20. Guardado final actualizado
Se guarda nuevamente el CSV limpio incluyendo las nuevas columnas de análisis emocional.


In [230]:
df_liverpool.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Archivo actualizado guardado en: {OUTPUT_FILE}")


Archivo actualizado guardado en: c:\Users\moise\OneDrive\DATA SCIENCE - TRIPLE TEN\LIVERPOOL\Dashboard_Liverpool_VN_Aniv_Oct_26\notebooks\data\processed\Liverpool_VN_Aniv_Oct_26_consolidado_2.0_limpio.csv
